misc notes
- response needs to have user idx, item idx, and relevance

# set up environment

In [6]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict
import shutil

from pyspark.sql import SparkSession
from pyspark.sql import functions as sf
from pyspark.sql import DataFrame, Window
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.linalg import Vectors, VectorUDT
from pyspark.sql.types import DoubleType, ArrayType

# Set up plotting
plt.style.use('seaborn-whitegrid')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)


/tmp/ipykernel_56791/45819036.py:18: MatplotlibDeprecationWarning: The seaborn styles shipped by Matplotlib are deprecated since 3.6, as they no longer correspond to the styles shipped by seaborn. However, they will remain available as 'seaborn-v0_8-<style>'. Alternatively, directly use the seaborn API instead.
  plt.style.use('seaborn-whitegrid')


# init spark session 

In [7]:
spark = (SparkSession.builder
    .appName("RecSysVisualization")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print("Spark session initialized.")

Spark session initialized.


In [8]:
sys.path.append(os.path.join(os.path.dirname(os.getcwd()), '..'))

from data_generator import CompetitionDataGenerator
from simulator import CompetitionSimulator
from sample_recommenders import (
    RandomRecommender,
    PopularityRecommender,
    ContentBasedRecommender
)
from config import DEFAULT_CONFIG, EVALUATION_METRICS

print("Competition modules imported.")

Competition modules imported.


In [46]:
from pyspark.ml.classification import LogisticRegression 
from pyspark.ml.feature import VectorAssembler, StringIndexer, OneHotEncoder, StandardScaler
from pyspark.ml import Pipeline 



class MyRecommender:
    """
    Template class for implementing a custom recommender.
    
    This class provides the basic structure required to implement a recommender
    that can be used with the Sim4Rec simulator. Students should extend this class
    with their own recommendation algorithm.
    """
    
    def __init__(self, seed=None):
        """
        Initialize recommender.
        
        Args:
            seed: Random seed for reproducibility
        """
        self.seed = seed
        # Add your initialization logic here
        self.model = None
    
    @staticmethod
    def one_hot_encode(df, column):
        pass

    def fit(self, log, user_features=None, item_features=None):
        """
        Train the recommender model based on interaction history.
        
        Args:
            log: Interaction log with user_idx, item_idx, and relevance columns
            user_features: User features dataframe (optional)
            item_features: Item features dataframe (optional)
        """
        # Implement your training logic here
        data = (log
                .join(user_features, on="user_idx", how="left")
                .join(item_features, on="item_idx", how="left")
                .drop("user_idx", "item_idx")
            )
        
        

        self.model = LogisticRegression(

        )
        
    
    def predict(self, log, k, users, items, user_features=None, item_features=None, filter_seen_items=True):
        """
        Generate recommendations for users.
        
        Args:
            log: Interaction log with user_idx, item_idx, and relevance columns
            k: Number of items to recommend
            users: User dataframe
            items: Item dataframe
            user_features: User features dataframe (optional)
            item_features: Item features dataframe (optional)
            filter_seen_items: Whether to filter already seen items
        
        Returns:
            DataFrame: Recommendations with user_idx, item_idx, and relevance columns
        """
        # Example of a random recommender implementation:
        # Cross join users and items
        recs = users.crossJoin(items)
        
        # Filter out already seen items if needed
        if filter_seen_items and log is not None:
            seen_items = log.select("user_idx", "item_idx")
            recs = recs.join(
                seen_items,
                on=["user_idx", "item_idx"],
                how="left_anti"
            )
        
        # Add random relevance scores
        recs = recs.withColumn(
            "relevance",
            sf.rand(seed=self.seed)
        )
        
        # Rank items by relevance for each user
        window = Window.partitionBy("user_idx").orderBy(sf.desc("relevance"))
        recs = recs.withColumn("rank", sf.row_number().over(window))
        
        # Filter top-k recommendations
        recs = recs.filter(sf.col("rank") <= k).drop("rank")
        
        return recs

print("MyRecommender template defined.")

MyRecommender template defined.


In [10]:
# We'll create a smaller dataset for demonstration.
config = DEFAULT_CONFIG.copy()
config['data_generation']['n_users'] = 1000  # Reduced from 10,000
config['data_generation']['n_items'] = 200   # Reduced from 1,000
config['data_generation']['seed'] = 42       # Fixed seed for reproducibility

# Get train-test split parameters
train_iterations = config['simulation']['train_iterations']
test_iterations = config['simulation']['test_iterations']

print(f"Running train-test simulation with {train_iterations} training iterations and {test_iterations} testing iterations")

# Initialize data generator
data_generator = CompetitionDataGenerator(
    spark_session=spark,
    **config['data_generation']
)

# Generate user data
users_df = data_generator.generate_users()
print(f"Generated {users_df.count()} users")

# Generate item data
items_df = data_generator.generate_items()
print(f"Generated {items_df.count()} items")

# Generate initial interaction history
history_df = data_generator.generate_initial_history(
    config['data_generation']['initial_history_density']
)
print(f"Generated {history_df.count()} initial interactions")

Running train-test simulation with 5 training iterations and 5 testing iterations


25/05/06 09:53:49 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


Generated 1000 users
Generated 200 items
Generated 200 initial interactions


## exploring data

In [11]:
import pprint
pprint.pprint(config)

{'competition': {'evaluation_iterations': 20,
                 'evaluation_metric': 'revenue',
                 'max_submission_per_day': 3,
                 'private_leaderboard_fraction': 0.6,
                 'public_leaderboard_fraction': 0.4,
                 'time_limit_seconds': 3600},
 'data_generation': {'initial_history_density': 0.001,
                     'item_categories': ['electronics',
                                         'books',
                                         'clothing',
                                         'home'],
                     'item_category_means': [0.3, -0.3, 0.0, 0.2],
                     'item_category_weights': [0.2, 0.3, 0.3, 0.2],
                     'item_feature_mean': 0.0,
                     'item_feature_std': 1.0,
                     'n_item_features': 20,
                     'n_items': 200,
                     'n_user_features': 20,
                     'n_users': 1000,
                     'seed': 42,
                  

In [12]:
print(type(users_df))
users_df.select('user_idx').count() - len(users_df.select('user_idx').distinct().collect())
# user idx is user id
users_df.show(5)
users_df.groupBy('segment').count().show()
users_df.describe().show()
# user features already normalized (enough)

<class 'pyspark.sql.dataframe.DataFrame'>
+--------------------+-------------------+--------------------+-------------------+-------------------+--------------------+-------------------+--------------------+--------------------+--------------------+--------------------+-------------------+-------------------+-------------------+-------------------+-------------------+--------------------+--------------------+--------------------+-------------------+--------+-------+
|         user_attr_0|        user_attr_1|         user_attr_2|        user_attr_3|        user_attr_4|         user_attr_5|        user_attr_6|         user_attr_7|         user_attr_8|         user_attr_9|        user_attr_10|       user_attr_11|       user_attr_12|       user_attr_13|       user_attr_14|       user_attr_15|        user_attr_16|        user_attr_17|        user_attr_18|       user_attr_19|user_idx|segment|
+--------------------+-------------------+--------------------+-------------------+-----------------

25/05/06 09:53:58 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+--------------------+--------------------+-------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+-------------------+--------------------+--------------------+--------------------+-----------------+-------+
|summary|         user_attr_0|         user_attr_1|        user_attr_2|         user_attr_3|         user_attr_4|         user_attr_5|         user_attr_6|         user_attr_7|         user_attr_8|         user_attr_9|        user_attr_10|        user_attr_11|        user_attr_12|        user_attr_13|        user_attr_14|        user_attr_15|       user_attr_16|        user_attr_17|        user_attr_18|        user_attr_19|         user_idx|segment|
+-------+--------------------+--------------------+-------------------+--------------------+

In [13]:
print(type(items_df))
items_df.show(5)
items_df.select('item_idx').count() - len(items_df.select('item_idx').distinct().collect())
# item idx is user id
items_df.groupBy('category').count().show()
items_df.describe().show()
# item_attr_.* are normalized, price is not

<class 'pyspark.sql.dataframe.DataFrame'>
+-------------------+--------------------+-------------------+--------------------+-------------------+--------------------+--------------------+-------------------+-------------------+--------------------+--------------------+--------------------+-------------------+-------------------+--------------------+-------------------+-------------------+--------------------+-------------------+-------------------+--------+-----------+------------------+
|        item_attr_0|         item_attr_1|        item_attr_2|         item_attr_3|        item_attr_4|         item_attr_5|         item_attr_6|        item_attr_7|        item_attr_8|         item_attr_9|        item_attr_10|        item_attr_11|       item_attr_12|       item_attr_13|        item_attr_14|       item_attr_15|       item_attr_16|        item_attr_17|       item_attr_18|       item_attr_19|item_idx|   category|             price|
+-------------------+--------------------+--------------

In [14]:
history_df.show(5)
history_df.groupBy('user_idx').count().groupBy('count').count().show()
# many one-time interactions
history_df.groupBy('item_idx').count().groupBy('count').count().show()
# a couple of popular items 

+--------+--------+---------+
|user_idx|item_idx|relevance|
+--------+--------+---------+
|     396|      97|        1|
|     336|      25|        0|
|     935|     102|        0|
|     967|     197|        0|
|     854|      96|        1|
+--------+--------+---------+
only showing top 5 rows

+-----+-----+
|count|count|
+-----+-----+
|    1|  162|
|    2|   19|
+-----+-----+

+-----+-----+
|count|count|
+-----+-----+
|    1|   81|
|    3|    4|
|    2|   41|
|    4|    5|
|    5|    1|
+-----+-----+



In [78]:
import re

data = (
    history_df.join(users_df, on="user_idx", how="left")
    .join(items_df, on="item_idx", how="left")
    .drop("user_idx", "item_idx")
)
data.show(5)

data_pipeline = Pipeline(
    stages=[
        categoryIDX := StringIndexer(inputCol="category", outputCol="category_idx"),
        segmentIDX := StringIndexer(inputCol="segment", outputCol="segment_idx"),
        categoryEncode := OneHotEncoder(
            inputCol="category_idx", outputCol="category_vec"
        ),
        segmentEncode := OneHotEncoder(inputCol="segment_idx", outputCol="segment_vec"),
        assembler := VectorAssembler(
            inputCols=
            [c for c in data.columns if re.match(r".*_attr_.*", c)] + 
                ["category_vec", "segment_vec", "price"],
            outputCol="features",
        ),
        # scaler := StandardScaler(
        #     inputCol="price",
        #     outputCol="price_scaled",
        # ),
    ]
)



model_pipeline = Pipeline(
    stages=[
        data_pipeline, 
        logistic := LogisticRegression(featuresCol="features", labelCol="relevance")
    ]
)

+---------+-------------------+-------------------+--------------------+-------------------+--------------------+------------------+-------------------+--------------------+-------------------+-------------------+-------------------+-------------------+--------------------+--------------------+--------------------+-------------------+-------------------+-------------------+--------------------+--------------------+----------+-------------------+-------------------+--------------------+-------------------+-------------------+-------------------+-------------------+-------------------+--------------------+-------------------+-------------------+--------------------+-------------------+-------------------+--------------------+--------------------+-------------------+-------------------+-------------------+--------------------+-----------+------------------+
|relevance|        user_attr_0|        user_attr_1|         user_attr_2|        user_attr_3|         user_attr_4|       user_attr_5| 

In [ ]:
train_data, test_data = data.randomSplit([0.7, 0.3]) 
fit_model = model_pipeline.fit(train_data)
predictions = fit_model.transform(test_data)
predictions.show(5)

+---------+-------------------+-------------------+-------------------+--------------------+-------------------+-------------------+--------------------+--------------------+-------------------+--------------------+--------------------+-------------------+--------------------+--------------------+--------------------+-------------------+--------------------+--------------------+-------------------+--------------------+----------+-------------------+-------------------+--------------------+-------------------+-------------------+-------------------+-------------------+--------------------+--------------------+--------------------+--------------------+-------------------+-------------------+-------------------+-------------------+--------------------+-------------------+-------------------+-------------------+-------------------+-----------+------------------+------------+-----------+-------------+-------------+--------------------+--------------------+--------------------+----------+
|r

In [86]:
predictions.select('relevance rawPrediction probability prediction'.split()).show(10, truncate=False)

+---------+----------------------------------------+----------------------------------------+----------+
|relevance|rawPrediction                           |probability                             |prediction|
+---------+----------------------------------------+----------------------------------------+----------+
|0        |[2.180475622880903,-2.180475622880903]  |[0.8984824627095798,0.10151753729042023]|0.0       |
|0        |[-2.052310283663009,2.052310283663009]  |[0.1138191480367812,0.8861808519632188] |1.0       |
|0        |[-1.6281669475899712,1.6281669475899712]|[0.1640816257883166,0.8359183742116834] |1.0       |
|0        |[2.5014872885180024,-2.5014872885180024]|[0.9242460186814789,0.07575398131852107]|0.0       |
|0        |[-2.4618298053463468,2.4618298053463468]|[0.07857775104399345,0.9214222489560066]|1.0       |
|1        |[-1.4816838832828556,1.4816838832828556]|[0.18517321338621331,0.8148267866137867]|1.0       |
|0        |[-2.1877649481352774,2.1877649481352774]|[0.

In [82]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator 
res = BinaryClassificationEvaluator(
    rawPredictionCol='prediction',labelCol='relevance', metricName='areaUnderROC'
    ).evaluate(predictions) 
print(f"Area under ROC: {res}")

Area under ROC: 0.4766483516483517


# setting up generators and recommenders

In [40]:
user_generator, item_generator = data_generator.setup_data_generators()


In [41]:
print(type(user_generator))

<class 'sim4rec.modules.generator.CompositeGenerator'>


In [43]:

# Initialize the recommenders we want to compare
recommenders = [
    RandomRecommender(seed=42),
    PopularityRecommender(alpha=1.0, seed=42),
    ContentBasedRecommender(similarity_threshold=0.0, seed=42),
]
recommender_names = ["Random", "Popularity", "ContentBased", "MyRecommender"]

# Fit each recommender on the initial history
for recommender in recommenders:
    recommender.fit(log=data_generator.history_df)

print("Recommenders set up and initial fit complete.")

Recommenders set up and initial fit complete.


In [53]:
user_generator.sample(0.8).show(5)
item_generator.sample(0.8).show(5)

+--------------------+-------------------+--------------------+-------------------+--------------------+--------------------+--------------------+-------------------+-------------------+--------------------+-------------------+-------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+-------------------+-------------------+--------+-------+
|         user_attr_0|        user_attr_1|         user_attr_2|        user_attr_3|         user_attr_4|         user_attr_5|         user_attr_6|        user_attr_7|        user_attr_8|         user_attr_9|       user_attr_10|       user_attr_11|        user_attr_12|        user_attr_13|        user_attr_14|        user_attr_15|        user_attr_16|        user_attr_17|       user_attr_18|       user_attr_19|user_idx|segment|
+--------------------+-------------------+--------------------+-------------------+--------------------+--------------------+-------------